# 01 · Data Acquisition & Profiling

**Project:** AI-Powered Inventory Performance & Optimization for a Global Industrial Supply Chain

This notebook is the entry point of the pipeline. It:

1. Downloads the **Supply Chain Logistics Problem Dataset** (Brunel University London) — the real multi-plant/warehouse/port network backbone.
2. Downloads **Online Retail II** (UCI ML Repository) — the real demand-forecasting module.
3. Profiles both (shape, dtypes, nulls, date ranges, key cardinalities).
4. Saves cleaned Parquet copies to `data/processed/` for every downstream notebook to reuse.

Run this in **Google Colab** or any environment with open internet access — it downloads straight from the original sources. If a direct download is blocked by your network, it automatically falls back to the local copies already checked into `data/raw/` (see `docs/DATA_PROVENANCE.md` for exactly what those fallback copies are and their limits).

Sources, licenses and required attribution: see `docs/DATA_PROVENANCE.md`.

In [1]:
import os
import io
import zipfile
import urllib.request
from pathlib import Path

import pandas as pd

# Resolve project root whether we're run from notebooks/ locally or from a
# fresh Colab clone of the repo.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

BRUNEL_DIR = RAW_DIR / "brunel_logistics"
RETAIL_DIR = RAW_DIR / "online_retail"
RETAIL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: /home/claude/inventory-intelligence-gsc


## 1. Supply Chain Logistics Problem Dataset (Brunel University London)

Original source: https://brunel.figshare.com/articles/dataset/Supply_Chain_Logistics_Problem_Dataset/7558679
License: CC BY 4.0 (confirm the badge on the page before publishing — see `docs/DATA_PROVENANCE.md`).
Required attribution: Kalganova, T. & Dzalbs, I., "Supply Chain Logistics Problem Dataset," Brunel University London (Figshare).

In [2]:
FIGSHARE_ARTICLE_ID = 7558679
FIGSHARE_API = f"https://api.figshare.com/v2/articles/{FIGSHARE_ARTICLE_ID}"

def download_brunel_from_figshare(dest_dir: Path) -> bool:
    """Try to pull the original Excel file straight from Figshare's API.
    Returns True on success, False if the network/host is unreachable
    (e.g. this notebook is running in a sandboxed environment that blocks
    figshare.com — in that case we fall back to the local mirror copy)."""
    try:
        import json
        with urllib.request.urlopen(FIGSHARE_API, timeout=15) as r:
            meta = json.loads(r.read())
        for f in meta.get("files", []):
            name = f["name"]
            url = f["download_url"]
            out_path = dest_dir / name
            urllib.request.urlretrieve(url, out_path)
            print("Downloaded:", out_path)
        return True
    except Exception as e:
        print(f"Direct Figshare download unavailable here ({e!r}); using local fallback copy.")
        return False

BRUNEL_DIR.mkdir(parents=True, exist_ok=True)
have_brunel = any(BRUNEL_DIR.glob("*.csv"))
if not have_brunel:
    download_brunel_from_figshare(BRUNEL_DIR)
else:
    print("Brunel CSVs already present in", BRUNEL_DIR, "- skipping download.")

brunel_tables = {p.stem: pd.read_csv(p) for p in sorted(BRUNEL_DIR.glob("*.csv"))}
for name, df in brunel_tables.items():
    print(f"{name:20s} rows={len(df):>6}  cols={df.shape[1]:>3}  -> {list(df.columns)}")

Brunel CSVs already present in /home/claude/inventory-intelligence-gsc/data/raw/brunel_logistics - skipping download.
FreightRates         rows=  1540  cols= 11  -> ['Carrier', 'Orig_Port', 'Dest_Port', 'Min_Weight_Quant', 'Max_Weight_Quant', 'Service_Level', 'Min_Cost', 'Rate', 'Mode_DSC', 'TPT_Day_Count', 'Carrier_Type']
OrderList            rows=  9215  cols= 14  -> ['Order_ID', 'Order_Date', 'Orig_Port', 'Carrier', 'TPT_Day_Count', 'Service_Level', 'Ship_Ahead_Day_Count', 'Ship_Late_Day_Count', 'Customer', 'Product_ID', 'Plant_Code', 'Dest_Port', 'Unit_Quant', 'Weight']
PlantPorts           rows=    22  cols=  2  -> ['Plant_Code', 'Ports']
ProductsPerPlant     rows=  2036  cols=  2  -> ['Plant_Code', 'Product_ID']
VmiCustomers         rows=    14  cols=  2  -> ['Plant_Code', 'Customer']
WhCapacities         rows=    19  cols=  2  -> ['Plant_Code', 'Daily_Capacity']
WhCosts              rows=    19  cols=  2  -> ['Plant_Code', 'Cost_Per_Unit']


## 2. Online Retail II (UCI Machine Learning Repository)

Original source: https://archive.ics.uci.edu/dataset/502/online+retail+ii
License: CC BY 4.0. Required attribution: Chen, D. (2019). *Online Retail II* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CG6D

**Note:** the file that ships in `data/raw/online_retail/` by default is a
development placeholder (see `docs/DATA_PROVENANCE.md`) used because the
build sandbox this project started in cannot reach `archive.ics.uci.edu`.
Running this cell somewhere with open internet access (Colab included)
replaces it with the authentic Online Retail II file automatically.

In [3]:
UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"

def download_online_retail_ii(dest_dir: Path) -> Path | None:
    try:
        with urllib.request.urlopen(UCI_ZIP_URL, timeout=20) as r:
            data = r.read()
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            z.extractall(dest_dir)
        xlsx_files = list(dest_dir.glob("*.xlsx"))
        print("Downloaded and extracted:", xlsx_files)
        return xlsx_files[0] if xlsx_files else None
    except Exception as e:
        print(f"Direct UCI download unavailable here ({e!r}); using local fallback copy.")
        return None

retail_xlsx = list(RETAIL_DIR.glob("*.xlsx"))
retail_source_note = ""
if retail_xlsx:
    retail_df = pd.concat(
        [pd.read_excel(retail_xlsx[0], sheet_name=s) for s in pd.ExcelFile(retail_xlsx[0]).sheet_names],
        ignore_index=True,
    )
    retail_source_note = "authentic Online Retail II (UCI #502)"
else:
    result = download_online_retail_ii(RETAIL_DIR)
    if result is not None:
        retail_df = pd.concat(
            [pd.read_excel(result, sheet_name=s) for s in pd.ExcelFile(result).sheet_names],
            ignore_index=True,
        )
        retail_source_note = "authentic Online Retail II (UCI #502)"
    else:
        placeholder = RETAIL_DIR / "online_retail_dev_placeholder.csv"
        retail_df = pd.read_csv(placeholder)
        retail_source_note = "DEV PLACEHOLDER — original 'Online Retail' (UCI #352), not II. Swap before publishing."

print("Using:", retail_source_note)
print("Shape:", retail_df.shape)
retail_df.head()

Direct UCI download unavailable here (URLError(OSError('Tunnel connection failed: 403 Forbidden'))); using local fallback copy.


Using: DEV PLACEHOLDER — original 'Online Retail' (UCI #352), not II. Swap before publishing.
Shape: (531282, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 3. Profile both sources

In [4]:
def profile(df: pd.DataFrame, name: str):
    print(f"=== {name} ===")
    print("shape:", df.shape)
    print("nulls per column:")
    print(df.isna().sum())
    print()

for name, df in brunel_tables.items():
    profile(df, name)

profile(retail_df, "online_retail")

if "InvoiceDate" in retail_df.columns:
    retail_df["InvoiceDate"] = pd.to_datetime(retail_df["InvoiceDate"])
    print("Retail date range:", retail_df["InvoiceDate"].min(), "->", retail_df["InvoiceDate"].max())
    print("Unique StockCodes:", retail_df["StockCode"].nunique())
    print("Unique Countries:", retail_df["Country"].nunique())

if "OrderList" in brunel_tables:
    ol = brunel_tables["OrderList"]
    print("Brunel unique Product_ID:", ol["Product_ID"].nunique())
    print("Brunel unique Plant_Code:", ol["Plant_Code"].nunique())
    print("Brunel unique Dest_Port:", ol["Dest_Port"].nunique())

=== FreightRates ===
shape: (1540, 11)
nulls per column:
Carrier             0
Orig_Port           0
Dest_Port           0
Min_Weight_Quant    0
Max_Weight_Quant    0
Service_Level       0
Min_Cost            0
Rate                0
Mode_DSC            0
TPT_Day_Count       0
Carrier_Type        0
dtype: int64

=== OrderList ===
shape: (9215, 14)
nulls per column:
Order_ID                0
Order_Date              0
Orig_Port               0
Carrier                 0
TPT_Day_Count           0
Service_Level           0
Ship_Ahead_Day_Count    0
Ship_Late_Day_Count     0
Customer                0
Product_ID              0
Plant_Code              0
Dest_Port               0
Unit_Quant              0
Weight                  0
dtype: int64

=== PlantPorts ===
shape: (22, 2)
nulls per column:
Plant_Code    0
Ports         0
dtype: int64

=== ProductsPerPlant ===
shape: (2036, 2)
nulls per column:
Plant_Code    0
Product_ID    0
dtype: int64

=== VmiCustomers ===
shape: (14, 2)
nulls per colum

## 4. Save cleaned copies to `data/processed/` for every downstream notebook

In [5]:
for name, df in brunel_tables.items():
    df.to_parquet(PROCESSED_DIR / f"brunel_{name}.parquet", index=False)

retail_clean = retail_df.dropna(subset=["CustomerID"]).copy()
retail_clean = retail_clean[retail_clean["Quantity"] > 0]
retail_clean = retail_clean[retail_clean["UnitPrice"] > 0]
retail_clean["LineTotal"] = retail_clean["Quantity"] * retail_clean["UnitPrice"]
retail_clean.to_parquet(PROCESSED_DIR / "online_retail_clean.parquet", index=False)

print("Saved processed tables to", PROCESSED_DIR)
print("Retail rows after basic cleaning:", len(retail_clean), "of", len(retail_df))
print("Source used for this run:", retail_source_note)

Saved processed tables to /home/claude/inventory-intelligence-gsc/data/processed
Retail rows after basic cleaning: 530103 of 531282
Source used for this run: DEV PLACEHOLDER — original 'Online Retail' (UCI #352), not II. Swap before publishing.
